In [17]:
from pathlib import Path
from io import StringIO
import requests
import pandas as pd
import numpy as np
import json
import zipfile
import io
import time
from tqdm.notebook import tqdm

In [18]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

EARTHQUAKE_DIR = RAW_DIR / "earthquakes"
SHAKEMAP_DIR = RAW_DIR / "shakemap"
GEOLOGY_DIR = RAW_DIR / "geology"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

for directory in [
    EARTHQUAKE_DIR,
    SHAKEMAP_DIR,
    GEOLOGY_DIR,
    INTERIM_DIR,
    PROCESSED_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

Project root:
C:\Documents\Earthquake_ML_DL


In [19]:
START_DATE = "2000-01-01"
END_DATE = "2026-08-09"

MIN_LATITUDE = 5
MAX_LATITUDE = 38

MIN_LONGITUDE = 65
MAX_LONGITUDE = 100

MIN_MAGNITUDE = 4.0

USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

print("Configuration loaded.")

Configuration loaded.


In [20]:
params = {
    "format": "csv",
    "starttime": START_DATE,
    "endtime": END_DATE,

    "minlatitude": MIN_LATITUDE,
    "maxlatitude": MAX_LATITUDE,

    "minlongitude": MIN_LONGITUDE,
    "maxlongitude": MAX_LONGITUDE,

    "minmagnitude": MIN_MAGNITUDE,

    "eventtype": "earthquake",

    "orderby": "time-asc",
    "limit": 20000
}

response = requests.get(
    USGS_URL,
    params=params,
    timeout=120
)

response.raise_for_status()

print("Download successful")
print(f"Downloaded: {len(response.content) / 1024:.1f} KB")

Download successful
Downloaded: 3118.2 KB


In [21]:
earthquakes = pd.read_csv(
    StringIO(response.text)
)

print("Shape:", earthquakes.shape)

earthquakes.head()

Shape: (18339, 22)


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2000-01-01T05:24:35.290Z,36.874,69.947,54.3,5.1,mwc,NaN,NaN,NaN,0.83,...,2022-04-29T18:27:48.504Z,"29 km SSE of Rust?q, Afghanistan",earthquake,NaN,12.7,NaN,NaN,reviewed,us,hrv
1,2000-01-01T06:26:04.210Z,37.027,69.964,33.0,4.4,mb,NaN,NaN,NaN,1.25,...,2014-11-07T01:09:15.130Z,"16 km SE of Rust?q, Afghanistan",earthquake,NaN,NaN,NaN,7.0,reviewed,us,us
2,2000-01-02T10:21:17.550Z,36.229,70.893,33.0,4.1,mb,NaN,NaN,NaN,0.90,...,2014-11-07T01:09:16.101Z,"70 km S of Jurm, Afghanistan",earthquake,NaN,NaN,NaN,4.0,reviewed,us,us
3,2000-01-02T10:23:58.980Z,27.559,92.498,33.0,5.0,mb,NaN,NaN,NaN,0.86,...,2022-04-29T19:37:55.502Z,"33 km NNE of Bomdila, India",earthquake,NaN,NaN,NaN,52.0,reviewed,us,us
4,2000-01-03T22:34:12.640Z,22.132,92.771,33.0,4.6,mb,NaN,NaN,NaN,1.08,...,2021-10-20T18:04:49.845Z,"45 km SSW of Saiha, India",earthquake,NaN,NaN,NaN,10.0,reviewed,us,us


In [22]:
columns = [
    "time",
    "latitude",
    "longitude",
    "depth",
    "mag",
    "magType",
    "place",
    "type",
    "status",
    "net",
    "id"
]

earthquakes = earthquakes[columns].copy()

In [23]:
earthquakes["time"] = pd.to_datetime(
    earthquakes["time"],
    errors="coerce"
)

earthquakes = earthquakes.dropna(
    subset=[
        "time",
        "latitude",
        "longitude",
        "depth",
        "mag"
    ]
)

earthquakes = earthquakes.drop_duplicates(
    subset="id"
)

earthquakes = earthquakes.sort_values(
    "time"
).reset_index(drop=True)

print("Final earthquake records:", len(earthquakes))

Final earthquake records: 18339


In [25]:
earthquake_file = (
    EARTHQUAKE_DIR /
    "india_region_earthquakes_2000_2026.csv"
)

earthquakes.to_csv(
    earthquake_file,
    index=False
)

print(f"Saved:\n{earthquake_file}")

Saved:
C:\Documents\Earthquake_ML_DL\data\raw\earthquakes\india_region_earthquakes_2000_2026.csv


In [26]:
print(f"Total earthquakes: {len(earthquakes):,}")

print("\nMagnitude distribution:")

magnitude_bins = [
    4.0, 4.5, 5.0, 5.5,
    6.0, 6.5, 7.0, 7.5,
    8.0, 10.0
]

magnitude_counts = (
    earthquakes["mag"]
    .groupby(
        pd.cut(
            earthquakes["mag"],
            bins=magnitude_bins,
            include_lowest=True
        )
    )
    .size()
)

print(magnitude_counts)

Total earthquakes: 18,339

Magnitude distribution:
mag
(3.999, 4.5]    11957
(4.5, 5.0]       4814
(5.0, 5.5]       1209
(5.5, 6.0]        247
(6.0, 6.5]         74
(6.5, 7.0]         22
(7.0, 7.5]         10
(7.5, 8.0]          6
Name: mag, dtype: int64


In [27]:
for threshold in [4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0]:
    count = (
        earthquakes["mag"] >= threshold
    ).sum()

    print(
        f"M >= {threshold:.1f}: {count:,}"
    )

M >= 4.0: 18,339
M >= 4.5: 8,362
M >= 5.0: 2,067
M >= 5.5: 482
M >= 6.0: 134
M >= 6.5: 43
M >= 7.0: 16


In [28]:
events_by_year = (
    earthquakes["time"]
    .dt.year
    .value_counts()
    .sort_index()
)

events_by_year

time
2000     356
2001     499
2002     499
2003     377
2004    1038
2005    2634
2006     770
2007     663
2008    1084
2009     426
2010     454
2011     381
2012     502
2013     608
2014     732
2015     944
2016     562
2017     459
2018     540
2019     597
2020     602
2021     636
2022     644
2023     639
2024     548
2025     799
2026     346
Name: count, dtype: int64

In [29]:
quality_report = pd.DataFrame({
    "dtype": earthquakes.dtypes.astype(str),
    "missing": earthquakes.isna().sum(),
    "unique": earthquakes.nunique()
})

quality_report

,dtype,missing,unique
time,"datetime64[us, UTC]",0,18339
latitude,float64,0,14883
longitude,float64,0,14512
depth,float64,0,5995
mag,float64,0,40
magType,str,0,9
place,str,0,10614
type,str,0,1
status,str,0,1
net,str,0,2


In [30]:
shakemap_params = {
    "format": "csv",

    "starttime": START_DATE,
    "endtime": END_DATE,

    "minlatitude": MIN_LATITUDE,
    "maxlatitude": MAX_LATITUDE,

    "minlongitude": MIN_LONGITUDE,
    "maxlongitude": MAX_LONGITUDE,

    "minmagnitude": MIN_MAGNITUDE,

    "eventtype": "earthquake",

    "producttype": "shakemap",

    "orderby": "time-asc",
    "limit": 20000
}

shakemap_response = requests.get(
    USGS_URL,
    params=shakemap_params,
    timeout=120
)

shakemap_response.raise_for_status()

print("ShakeMap query successful")
print(
    f"Downloaded: "
    f"{len(shakemap_response.content) / 1024:.1f} KB"
)

ShakeMap query successful
Downloaded: 106.5 KB


In [31]:
shakemap_events = pd.read_csv(
    StringIO(shakemap_response.text)
)

print(
    "ShakeMap event records:",
    len(shakemap_events)
)

shakemap_events.head()

ShakeMap event records: 635


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2000-01-03T22:34:12.640Z,22.132,92.771,33.0,4.6,mb,NaN,NaN,NaN,1.08,...,2021-10-20T18:04:49.845Z,"45 km SSW of Saiha, India",earthquake,NaN,NaN,NaN,10.0,reviewed,us,us
1,2000-01-05T09:45:18.710Z,32.222,92.700,33.0,5.5,mwc,NaN,NaN,NaN,0.82,...,2022-04-29T18:28:10.339Z,"102 km NE of Nagqu, China",earthquake,NaN,NaN,NaN,NaN,reviewed,us,hrv
2,2000-01-19T07:09:33.580Z,36.372,70.379,206.9,6.0,mwb,NaN,NaN,NaN,0.91,...,2022-04-29T18:28:41.311Z,"51 km ESE of Farkh?r, Afghanistan",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us
3,2000-03-12T18:03:56.270Z,17.099,73.672,33.0,5.0,mwc,NaN,NaN,NaN,1.08,...,2022-04-29T18:34:39.517Z,"26 km SE of M?khjan, India",earthquake,NaN,NaN,NaN,NaN,reviewed,us,hrv
4,2000-03-31T14:13:31.470Z,28.758,70.004,33.0,4.5,mb,NaN,NaN,NaN,1.08,...,2020-04-16T18:12:52.941Z,"9 km NNE of Rojhan, Pakistan",earthquake,NaN,NaN,NaN,26.0,reviewed,us,us


In [32]:
total_events = len(earthquakes)
shakemap_count = len(shakemap_events)

percentage = (
    shakemap_count /
    total_events *
    100
)

print(f"Total earthquake events : {total_events:,}")
print(f"ShakeMap events         : {shakemap_count:,}")
print(f"ShakeMap coverage       : {percentage:.2f}%")

Total earthquake events : 18,339
ShakeMap events         : 635
ShakeMap coverage       : 3.46%


In [33]:
shakemap_ids = set(
    shakemap_events["id"].dropna()
)

print(
    f"Unique ShakeMap event IDs: "
    f"{len(shakemap_ids):,}"
)

Unique ShakeMap event IDs: 635


In [34]:
shake_events = earthquakes[
    earthquakes["id"].isin(shakemap_ids)
].copy()

shake_events = shake_events.reset_index(drop=True)

print(
    "Earthquake events with ShakeMap:",
    len(shake_events)
)

Earthquake events with ShakeMap: 635


In [35]:
shakemap_event_file = (
    EARTHQUAKE_DIR /
    "shakemap_available_events.csv"
)

shake_events.to_csv(
    shakemap_event_file,
    index=False
)

print(
    f"Saved:\n{shakemap_event_file}"
)

Saved:
C:\Documents\Earthquake_ML_DL\data\raw\earthquakes\shakemap_available_events.csv


In [36]:
def get_event_detail(event_id):
    
    url = (
        "https://earthquake.usgs.gov/"
        "earthquakes/feed/v1.0/detail/"
        f"{event_id}.geojson"
    )

    try:
        response = requests.get(
            url,
            timeout=30
        )

        if response.status_code != 200:
            return None

        return response.json()

    except requests.RequestException:
        return None

In [37]:
test_event_id = shake_events.iloc[0]["id"]

detail = get_event_detail(
    test_event_id
)

print(
    "Event:",
    test_event_id
)

print(
    "Retrieved:",
    detail is not None
)

Event: usp0009kqm
Retrieved: True


In [38]:
if detail is not None:

    products = (
        detail
        .get("properties", {})
        .get("products", {})
    )

    print(
        "Available product types:"
    )

    print(
        list(products.keys())
    )

Available product types:
['impact-text', 'origin', 'phase-data', 'shakemap']


In [39]:
if detail is not None:

    shakemap_products = (
        detail
        .get("properties", {})
        .get("products", {})
        .get("shakemap", [])
    )

    print(
        "Number of ShakeMap products:",
        len(shakemap_products)
    )

    if shakemap_products:
        print(
            shakemap_products[0].keys()
        )

Number of ShakeMap products: 1
dict_keys(['indexid', 'indexTime', 'id', 'type', 'code', 'source', 'updateTime', 'status', 'properties', 'preferredWeight', 'contents'])


In [40]:
def extract_shakemap_product(detail):

    if detail is None:
        return None

    products = (
        detail
        .get("properties", {})
        .get("products", {})
    )

    shakemap_products = products.get(
        "shakemap",
        []
    )

    if not shakemap_products:
        return None

    # Prefer the latest product
    shakemap_products = sorted(
        shakemap_products,
        key=lambda x: x.get("updateTime", 0),
        reverse=True
    )

    product = shakemap_products[0]

    contents = product.get(
        "contents",
        {}
    )

    # Prefer raw XYZ grid
    preferred_files = [
        "download/grid.xyz.zip",
        "download/grid.xml.zip",
        "grid.xyz",
        "grid.xml"
    ]

    for filename in preferred_files:

        if filename in contents:

            return {
                "event_id": detail["id"],
                "update_time": product.get(
                    "updateTime"
                ),
                "source": product.get(
                    "source"
                ),
                "version": product.get(
                    "version"
                ),
                "url": contents[filename].get(
                    "url"
                ),
                "filename": filename
            }

    return None

In [41]:
product_info = extract_shakemap_product(
    detail
)

product_info

In [43]:
product_records = []

for event_id in tqdm(
    shake_events["id"],
    desc="Finding ShakeMap products"
):
    detail = get_event_detail(event_id)

    product = extract_shakemap_product(detail)

    if product is not None:
        product_records.append(product)

    time.sleep(0.05)

print(f"Products found: {len(product_records):,}")

Finding ShakeMap products:   0%|          | 0/635 [00:00<?, ?it/s]

Products found: 40


In [44]:
product_inventory = pd.DataFrame(
    product_records
)

print("Shape:", product_inventory.shape)

product_inventory.head()

Shape: (40, 6)


,event_id,update_time,source,version,url,filename
0,usc000ff4h,1364340210594,us,None,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip
1,usb000fzn7,1367582401348,us,None,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip
2,usb000g112,1366372786546,us,None,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip
3,usb000hdu8,1371128314208,us,None,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip
4,usc000kmdj,1388709976346,us,None,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip


In [45]:
print(
    f"Earthquake events: {len(earthquakes):,}"
)

print(
    f"ShakeMap events: {len(shake_events):,}"
)

print(
    f"Usable ShakeMap products: "
    f"{len(product_inventory):,}"
)

Earthquake events: 18,339
ShakeMap events: 635
Usable ShakeMap products: 40


In [46]:
product_inventory_file = (
    SHAKEMAP_DIR /
    "shakemap_product_inventory.csv"
)

product_inventory.to_csv(
    product_inventory_file,
    index=False
)

print(
    f"Saved:\n{product_inventory_file}"
)

Saved:
C:\Documents\Earthquake_ML_DL\data\raw\shakemap\shakemap_product_inventory.csv


In [47]:
product_inventory["source"].value_counts()

source
us    40
Name: count, dtype: int64

In [48]:
product_inventory["version"].value_counts()

Series([], Name: count, dtype: int64)

In [49]:
def get_file_size(url):

    try:
        r = requests.head(
            url,
            allow_redirects=True,
            timeout=30
        )

        size = r.headers.get(
            "Content-Length"
        )

        if size is not None:
            return int(size)

    except requests.RequestException:
        pass

    return None

In [50]:
sizes = []

for url in tqdm(
    product_inventory["url"],
    desc="Estimating download size"
):
    
    sizes.append(
        get_file_size(url)
    )

    time.sleep(0.05)

product_inventory["size_bytes"] = sizes

product_inventory["size_MB"] = (
    product_inventory["size_bytes"] /
    (1024 ** 2)
)

product_inventory["size_MB"].describe()

Estimating download size:   0%|          | 0/40 [00:00<?, ?it/s]

count    37.000000
mean      0.414585
std       0.234844
min       0.125098
25%       0.294106
50%       0.308081
75%       0.406697
max       1.098855
Name: size_MB, dtype: float64

In [51]:
known_sizes = (
    product_inventory["size_bytes"]
    .dropna()
)

total_gb = (
    known_sizes.sum()
    / (1024 ** 3)
)

print(
    f"Known total download size: "
    f"{total_gb:.2f} GB"
)

print(
    f"Products with known size: "
    f"{len(known_sizes):,} / "
    f"{len(product_inventory):,}"
)

Known total download size: 0.01 GB
Products with known size: 37 / 40


In [52]:
product_inventory[
    [
        "event_id",
        "source",
        "version",
        "size_MB"
    ]
].sort_values(
    "size_MB",
    ascending=False
).head(20)

,event_id,source,version,size_MB
37,us2000cp4g,us,None,1.098855
39,us1000ggp5,us,None,1.019886
38,us2000d1a1,us,None,0.973162
27,us10003vry,us,None,0.777320
11,usb000qy82,us,None,0.753852
9,usc000njrq,us,None,0.716994
36,us100088sf,us,None,0.707393
15,usb000sfrw,us,None,0.414768
17,usc000sy0y,us,None,0.414368
32,us200082s5,us,None,0.406697


In [53]:
product_inventory.to_csv(
    product_inventory_file,
    index=False
)

print("Product inventory updated.")

Product inventory updated.


In [54]:
download_results = []

for _, row in tqdm(
    product_inventory.iterrows(),
    total=len(product_inventory),
    desc="Downloading ShakeMap grids"
):
    
    event_id = row["event_id"]
    url = row["url"]
    
    event_dir = SHAKEMAP_DIR / str(event_id)
    event_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    
    filename = Path(row["filename"]).name
    
    output_file = event_dir / filename
    
    # Skip if already downloaded
    if output_file.exists():
        status = "exists"
    
    else:
        try:
            r = requests.get(
                url,
                timeout=120
            )
            
            r.raise_for_status()
            
            output_file.write_bytes(
                r.content
            )
            
            status = "downloaded"
            
        except Exception as e:
            print(
                f"Failed: {event_id}"
            )
            print(e)
            status = "failed"
    
    download_results.append({
        "event_id": event_id,
        "url": url,
        "file": str(output_file),
        "status": status
    })

In [55]:
download_log = pd.DataFrame(
    download_results
)

download_log["status"].value_counts()

status
downloaded    40
Name: count, dtype: int64

In [56]:
download_log_file = (
    SHAKEMAP_DIR /
    "download_log.csv"
)

download_log.to_csv(
    download_log_file,
    index=False
)

print(
    f"Saved:\n{download_log_file}"
)

Saved:
C:\Documents\Earthquake_ML_DL\data\raw\shakemap\download_log.csv


In [57]:
def directory_size_mb(path):

    total = 0

    for file in path.rglob("*"):
        if file.is_file():
            total += file.stat().st_size

    return total / (1024 ** 2)


print(
    f"Earthquake data: "
    f"{directory_size_mb(EARTHQUAKE_DIR):.2f} MB"
)

print(
    f"ShakeMap data: "
    f"{directory_size_mb(SHAKEMAP_DIR):.2f} MB"
)

Earthquake data: 2.30 MB
ShakeMap data: 16.33 MB


In [58]:
files = []

for file in RAW_DIR.rglob("*"):
    
    if file.is_file():
        
        files.append({
            "file": str(
                file.relative_to(PROJECT_ROOT)
            ),
            "size_MB": (
                file.stat().st_size /
                (1024 ** 2)
            )
        })

raw_inventory = pd.DataFrame(files)

raw_inventory.sort_values(
    "size_MB",
    ascending=False
)

,file,size_MB
0,data\raw\earthquakes\india_region_earthquakes_...,2.223466
22,data\raw\shakemap\us2000cp4g\grid.xyz.zip,1.098855
16,data\raw\shakemap\us1000ggp5\grid.xyz.zip,1.019886
23,data\raw\shakemap\us2000d1a1\grid.xyz.zip,0.973162
8,data\raw\shakemap\us10003vry\grid.xyz.zip,0.777320
30,data\raw\shakemap\usb000qy82\grid.xyz.zip,0.753852
39,data\raw\shakemap\usc000njrq\grid.xyz.zip,0.716994
15,data\raw\shakemap\us100088sf\grid.xyz.zip,0.707393
33,data\raw\shakemap\usb000sfrw\grid.xyz.zip,0.414768
42,data\raw\shakemap\usc000sy0y\grid.xyz.zip,0.414368
